# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv 


In [2]:
import dask.dataframe as dd

c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
import glob

# Get the root directory from the environment variable
price_data_dir = os.getenv('PRICE_DATA')

search_pattern = os.path.join(price_data_dir, "**", "**","*.parquet")

# glob.glob(search_pattern, recursive=True)

parquet_files = glob.glob(search_pattern, recursive=True)

# print("parquet file names:", parquet_files)

df = dd.read_parquet(parquet_files).compute().reset_index()

c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(


In [4]:
glob.glob(search_pattern, recursive=True)

['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2004\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2004\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2005\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2005\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2006\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2006\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2007\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2007\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2008\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2008\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_200

In [5]:
df

,index,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year
0,137315,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,ACN,2001
1,137316,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,ACN,2001
2,137317,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,ACN,2001
3,137318,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,ACN,2001
4,137319,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,ACN,2001
...,...,...,...,...,...,...,...,...,...,...,...
1116751,88143,2020-03-26,4.06,4.53,3.88,4.51,4.510000,1668500.0,ZIXI.csv,ZIXI,2020
1116752,88144,2020-03-27,4.49,4.71,4.10,4.60,4.600000,1146800.0,ZIXI.csv,ZIXI,2020
1116753,88145,2020-03-30,4.83,4.87,4.44,4.64,4.640000,1212000.0,ZIXI.csv,ZIXI,2020
1116754,88146,2020-03-31,4.60,4.69,4.10,4.31,4.310000,1057200.0,ZIXI.csv,ZIXI,2020


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [6]:
import dask.dataframe as dd
import random

# Sample 20 files from your list
sampled_files = random.sample(parquet_files, 20)

# Load sampled parquet files
ddf = dd.read_parquet(sampled_files)

# Sort before groupby+shift
ddf = ddf.sort_values(by=["ticker", "Date"])

# Lag features
ddf["Close_lag_1"] = ddf.groupby("ticker")["Close"].shift(1)
ddf["Adj_Close_lag_1"] = ddf.groupby("ticker")["Adj Close"].shift(1)

# Returns
ddf["returns"] = (ddf["Close"] / ddf["Close_lag_1"]) - 1

# Intraday range
ddf["hi_lo_range"] = ddf["High"] - ddf["Low"]

# Now compute (no reset_index yet!)
df_feat = ddf.compute()

# Sort again if needed
# df = df.sort_values(by=["ticker", "Date"])


C:\Users\sanjb\AppData\Local\Temp\ipykernel_24268\4265770480.py:14: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  ddf["Close_lag_1"] = ddf.groupby("ticker")["Close"].shift(1)
C:\Users\sanjb\AppData\Local\Temp\ipykernel_24268\4265770480.py:15: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  ddf["Adj_Close_lag_1"] = ddf.groupby("ticker")["Adj Close"].shift(1)


In [8]:
df_feat

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
123033,2007-01-05,6.620000,6.900000,6.600000,6.900000,6.900000,8500.0,NGD.csv,NGD,2007,6.820000,6.820000,0.011730,0.300000
123035,2007-01-09,7.310000,7.690000,7.250000,7.690000,7.690000,9200.0,NGD.csv,NGD,2007,7.200000,7.200000,0.068056,0.440000
123039,2007-01-16,7.980000,8.000000,7.820000,7.820000,7.820000,6300.0,NGD.csv,NGD,2007,7.880000,7.880000,-0.007614,0.180000
123047,2007-01-26,7.620000,7.650000,7.180000,7.250000,7.250000,2800.0,NGD.csv,NGD,2007,7.550000,7.550000,-0.039735,0.470000
123058,2007-02-12,7.350000,7.350000,7.100000,7.170000,7.170000,4500.0,NGD.csv,NGD,2007,7.260000,7.260000,-0.012397,0.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
925,2017-09-25,172.800003,176.639999,172.800003,174.720001,174.720001,100.0,AKER.csv,AKER,2017,174.720001,174.720001,0.000000,3.839996
942,2017-10-18,172.800003,178.559998,163.199997,165.119995,165.119995,1100.0,AKER.csv,AKER,2017,168.960007,168.960007,-0.022727,15.360001
961,2017-11-14,149.759995,158.975998,144.000000,151.679993,151.679993,2400.0,AKER.csv,AKER,2017,143.615997,143.615997,0.056150,14.975998
977,2017-12-07,85.632004,97.536003,77.183998,87.744003,87.744003,900.0,AKER.csv,AKER,2017,84.480003,84.480003,0.038636,20.352005


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [9]:
# Write your code below.

df_roll = ddf.compute()  # dask df

df_roll = df_roll.sort_values(by=["ticker", "Date"]) 

df_roll["returns_ma_10"] = (
    df_roll.groupby("ticker")["returns"]
    .rolling(window=10, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

df_roll

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_ma_10
868,2017-07-05,230.399994,240.000000,230.399994,230.399994,230.399994,600.0,AKER.csv,AKER,2017,NaN,NaN,NaN,9.600006,NaN
869,2017-07-06,240.000000,240.000000,224.255997,225.600006,225.600006,500.0,AKER.csv,AKER,2017,230.399994,230.399994,-0.020833,15.744003,-0.020833
870,2017-07-07,230.399994,230.399994,220.800003,230.399994,230.399994,300.0,AKER.csv,AKER,2017,225.600006,225.600006,0.021277,9.599991,0.000222
871,2017-07-10,230.399994,239.039993,220.800003,230.399994,230.399994,600.0,AKER.csv,AKER,2017,230.399994,230.399994,0.000000,18.239990,0.000148
872,2017-07-11,230.399994,240.000000,220.800003,220.800003,220.800003,600.0,AKER.csv,AKER,2017,230.399994,230.399994,-0.041667,19.199997,-0.010306
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100465,1988-12-23,3.593750,3.625000,3.593750,3.625000,0.594356,67600.0,WST.csv,WST,1988,3.593750,0.589232,0.008696,0.031250,0.002013
100466,1988-12-27,3.593750,3.625000,3.593750,3.625000,0.594356,10400.0,WST.csv,WST,1988,3.625000,0.594356,0.000000,0.031250,0.002013
100467,1988-12-28,3.593750,3.781250,3.593750,3.781250,0.619975,74000.0,WST.csv,WST,1988,3.625000,0.594356,0.043103,0.187500,0.011587
100468,1988-12-29,3.781250,3.781250,3.750000,3.750000,0.614851,393200.0,WST.csv,WST,1988,3.781250,0.619975,-0.008264,0.031250,0.006131


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

Pandas supports features to stratify by groups and rolling function. 

It would have been better to do it in Dask as it would be managable to handle large datasets. However, Dask do not support analysing by groups and rolling function.  

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.